In [7]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import pandas as pd
import numpy as np
import xgboost as xgb
import catboost as cb
import joblib
import json
import optuna
import gc
import os
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

print("--- Step 1: Library Imports and Initial Setup ---")

# --- Global Constants & Paths ---
RANDOM_STATE = 42
N_SPLITS = 5
N_OPTUNA_TRIALS_ERROR = 25
COMPETITION_ALPHA = 0.1
DATA_PATH = './'
MODELS_PATH = './mean_models/'
NN_PREDS_PATH = './NN_model_predictions/'

# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    return np.mean(width + penalty_lower + penalty_upper)

# --- Load Raw Data ---
print("\n--- Step 2: Loading Raw Dataset and Test Files ---")
try:
    drop_cols = ['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm', 'view_otherwater', 'view_other']
    df_train = pd.read_csv(os.path.join(DATA_PATH, 'dataset.csv')).drop(columns=drop_cols)
    df_test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv')).drop(columns=drop_cols)
    
    y_true = df_train['sale_price'].copy()
    grade_for_stratify = df_train['grade'].copy()
    
    print("Raw data loaded successfully. 'y_true' and 'grade_for_stratify' are ready.")
except FileNotFoundError as e:
    print(f"ERROR: Could not find data files. {e}")



--- Step 1: Library Imports and Initial Setup ---

--- Step 2: Loading Raw Dataset and Test Files ---
Raw data loaded successfully. 'y_true' and 'grade_for_stratify' are ready.


In [8]:
# Make sure to have these libraries installed
# pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import gc

# Define a random state for reproducibility
RANDOM_STATE = 42

def create_comprehensive_features(df_train, df_test):
    """
    Combines original and new advanced feature engineering steps into a single pipeline.
    """
    print("--- Starting Comprehensive Feature Engineering ---")

    # Store original indices and target variable
    train_ids = df_train.index
    test_ids = df_test.index
    y_train = df_train['sale_price'].copy() # Keep the target separate

    # Combine for consistent processing
    df_train_temp = df_train.drop(columns=['sale_price'])
    all_data = pd.concat([df_train_temp, df_test], axis=0, ignore_index=True)

    # --- Original Feature Engineering ---

    # A) Brute-Force Numerical Interactions
    print("Step 1: Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    # Ensure all columns exist and are numeric, fill missing with 0 for safety
    for col in NUMS:
        if col not in all_data.columns:
            all_data[col] = 0
        else:
            all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)
            
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # B) Date Features
    print("Step 2: Creating date features...")
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['sale_month'] = all_data['sale_date'].dt.month
    all_data['sale_dayofyear'] = all_data['sale_date'].dt.dayofyear
    all_data['age_at_sale'] = all_data['sale_year'] - all_data['year_built']

    # C) TF-IDF Text Features
    print("Step 3: Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        all_data = pd.concat([all_data, tfidf_df], axis=1)

    # D) Log transform some interaction features
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            all_data[c] = np.log1p(all_data[c].fillna(0))

    # --- New Feature Engineering Ideas ---

    # F) Group-By Aggregation Features
    print("Step 4: Creating group-by aggregation features...")
    group_cols = ['submarket', 'city', 'zoning']
    num_cols_for_agg = ['grade', 'sqft', 'imp_val', 'land_val', 'age_at_sale']

    for group_col in group_cols:
        for num_col in num_cols_for_agg:
            agg_stats = all_data.groupby(group_col)[num_col].agg(['mean', 'std', 'max', 'min']).reset_index()
            agg_stats.columns = [group_col] + [f'{group_col}_{num_col}_{stat}' for stat in ['mean', 'std', 'max', 'min']]
            all_data = pd.merge(all_data, agg_stats, on=group_col, how='left')
            all_data[f'{num_col}_minus_{group_col}_mean'] = all_data[num_col] - all_data[f'{group_col}_{num_col}_mean']

    # G) Ratio Features
    print("Step 5: Creating ratio features...")
    # Add a small epsilon to prevent division by zero
    epsilon = 1e-6 
    all_data['total_val'] = all_data['imp_val'] + all_data['land_val']
    all_data['imp_val_to_land_val_ratio'] = all_data['imp_val'] / (all_data['land_val'] + epsilon)
    all_data['land_val_ratio'] = all_data['land_val'] / (all_data['total_val'] + epsilon)
    all_data['sqft_to_lot_ratio'] = all_data['sqft'] / (all_data['sqft_lot'] + epsilon)
    all_data['was_renovated'] = (all_data['year_reno'] > 0).astype(int)
    all_data['reno_age_at_sale'] = np.where(all_data['was_renovated'] == 1, all_data['sale_year'] - all_data['year_reno'], -1)

    # H) Geospatial Clustering Features
    print("Step 6: Creating geospatial clustering features...")
    coords = all_data[['latitude', 'longitude']].copy()
    coords.fillna(coords.median(), inplace=True) # Simple imputation

    # KMeans is sensitive to feature scaling, but for lat/lon it's often okay without it.
    kmeans = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10) 
    all_data['location_cluster'] = kmeans.fit_predict(coords)
    
    # Calculate distance to each cluster center
    cluster_centers = kmeans.cluster_centers_
    for i in range(len(cluster_centers)):
        center = cluster_centers[i]
        all_data[f'dist_to_cluster_{i}'] = np.sqrt((coords['latitude'] - center[0])**2 + (coords['longitude'] - center[1])**2)

    # --- Final Cleanup ---
    print("Step 7: Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)

    # One-hot encode the new cluster feature
    all_data = pd.get_dummies(all_data, columns=['location_cluster'], prefix='loc_cluster')
    
    # Final check for any remaining object columns to be safe (besides index)
    object_cols = all_data.select_dtypes(include='object').columns
    if len(object_cols) > 0:
        print(f"Warning: Found unexpected object columns: {object_cols}. Dropping them.")
        all_data = all_data.drop(columns=object_cols)
        
    all_data.fillna(0, inplace=True)

    # Separate back into train and test sets
    train_len = len(train_ids)
    X = all_data.iloc[:train_len].copy()
    X_test = all_data.iloc[train_len:].copy()
    
    # Restore original indices
    X.index = train_ids
    X_test.index = test_ids
    
    # Align columns - crucial for model prediction
    X_test = X_test[X.columns]
    
    print(f"\nComprehensive FE complete. Total features: {X.shape[1]}")
    gc.collect()
    
    return X, X_test, y_train
# =============================================================================
# BLOCK 2.5: EXECUTE FEATURE ENGINEERING
# =============================================================================
print("\n--- Starting Block 2.5: Executing Feature Engineering Pipeline ---")

# This is the crucial step that was missing.
# We call the function to create our training and testing dataframes.
X, X_test, y_train = create_comprehensive_features(df_train, df_test)

# Let's verify the output
print(f"Feature engineering complete. X shape: {X.shape}, X_test shape: {X_test.shape}")
gc.collect()


--- Starting Block 2.5: Executing Feature Engineering Pipeline ---
--- Starting Comprehensive Feature Engineering ---
Step 1: Creating brute-force numerical interaction features...
Step 2: Creating date features...
Step 3: Creating TF-IDF features for text columns...
Step 4: Creating group-by aggregation features...
Step 5: Creating ratio features...
Step 6: Creating geospatial clustering features...
Step 7: Finalizing feature set...

Comprehensive FE complete. Total features: 233
Feature engineering complete. X shape: (200000, 233), X_test shape: (200000, 233)


0

In [9]:
# =============================================================================
# BLOCK 3: GENERATE & VALIDATE ALL MEAN PREDICTIONS (WITH FIXES)
# =============================================================================
print("\n--- Starting Block 3: Generating All Mean Predictions with Integrity Checks ---")

# --- A. Load Parameters & Initialize Arrays ---
print("\nStep A: Loading hyperparameters and initializing arrays...")
with open(os.path.join(MODELS_PATH, 'xgboost_best_params.json'), 'r') as f: best_params_xgb = json.load(f)
with open(os.path.join(MODELS_PATH, 'catboost_best_params.json'), 'r') as f: best_params_cb = json.load(f)
oof_xgb_preds, test_xgb_preds = np.zeros(len(X)), np.zeros(len(X_test))
oof_catboost_preds, test_catboost_preds = np.zeros(len(X)), np.zeros(len(X_test))

# --- B. Generate OOF Predictions via K-Fold Loop ---
print("\nStep B: Running K-Fold cross-prediction...")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Processing Fold {fold+1}/{N_SPLITS}...")
    X_train, y_train_fold = X.iloc[train_idx], y_true.iloc[train_idx]
    X_val, y_val_fold = X.iloc[val_idx], y_true.iloc[val_idx]
    
    # XGBoost
    bst = xgb.train(best_params_xgb, xgb.DMatrix(X_train, label=y_train_fold), num_boost_round=3000, evals=[(xgb.DMatrix(X_val, y_val_fold), 'v')], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb_preds[val_idx] = bst.predict(xgb.DMatrix(X_val), iteration_range=(0, bst.best_iteration))
    # CRITICAL FIX: Re-create the test DMatrix inside the loop to ensure feature consistency
    test_xgb_preds += bst.predict(xgb.DMatrix(X_test), iteration_range=(0, bst.best_iteration)) / N_SPLITS
    
    # CatBoost
    cb_model = cb.CatBoostRegressor(**best_params_cb).fit(X_train, y_train_fold, eval_set=[(X_val, y_val_fold)], early_stopping_rounds=100, use_best_model=True, verbose=0)
    oof_catboost_preds[val_idx] = cb_model.predict(X_val)
    test_catboost_preds += cb_model.predict(X_test) / N_SPLITS

# --- C. Load Neural Network Predictions ---
print("\nStep C: Loading pre-computed Neural Network predictions...")
oof_nn_preds = np.load(os.path.join(NN_PREDS_PATH, 'oof_nn_preds.npy'))
test_nn_preds = np.load(os.path.join(NN_PREDS_PATH, 'test_nn_preds.npy'))

# --- D. FORENSIC ANALYSIS: Check All Prediction Arrays ---
print("\nStep D: FORENSIC ANALYSIS of all generated predictions...")
for name, arr in [("OOF XGB", oof_xgb_preds), ("Test XGB", test_xgb_preds), 
                  ("OOF CB", oof_catboost_preds), ("Test CB", test_catboost_preds),
                  ("OOF NN", oof_nn_preds), ("Test NN", test_nn_preds)]:
    if np.isnan(arr).any() or np.isinf(arr).any():
        print(f"*** CRITICAL WARNING: Bad values (NaN/inf) found in '{name}' array! ***")
    else:
        print(f"  - '{name}' array is clean. Mean: {arr.mean():,.2f}, Std: {arr.std():,.2f}")

print("\nAll mean model predictions are now ready.")


--- Starting Block 3: Generating All Mean Predictions with Integrity Checks ---

Step A: Loading hyperparameters and initializing arrays...

Step B: Running K-Fold cross-prediction...
  Processing Fold 1/5...
  Processing Fold 2/5...
  Processing Fold 3/5...
  Processing Fold 4/5...
  Processing Fold 5/5...

Step C: Loading pre-computed Neural Network predictions...

Step D: FORENSIC ANALYSIS of all generated predictions...
  - 'OOF XGB' array is clean. Mean: 584,314.16, Std: 403,815.88
  - 'Test XGB' array is clean. Mean: 592,804.61, Std: 409,733.80
  - 'OOF CB' array is clean. Mean: 584,115.43, Std: 404,945.85
  - 'Test CB' array is clean. Mean: 593,032.59, Std: 411,630.22
  - 'OOF NN' array is clean. Mean: 583,830.59, Std: 402,890.90
  - 'Test NN' array is clean. Mean: -89,787,064.99, Std: 40,418,839,145.54

All mean model predictions are now ready.


In [12]:
# =============================================================================
# BLOCK 4 & 5: OPTIMIZED ENSEMBLE & ERROR MODEL
# =============================================================================
print("\n--- Starting Block 4: Optimizing 3-Model Ensemble Weights ---")
def get_ensemble_rmse(w): return np.sqrt(mean_squared_error(y_true, w[0]*oof_xgb_preds + w[1]*oof_catboost_preds + w[2]*oof_nn_preds))
res = minimize(get_ensemble_rmse, [1/3]*3, method='SLSQP', bounds=[(0,1)]*3, constraints={'type':'eq','fun':lambda w:1-np.sum(w)})
best_weights = res.x
print(f"Optimal Weights -> XGB: {best_weights[0]:.3f}, CB: {best_weights[1]:.3f}, NN: {best_weights[2]:.3f}")

oof_ensemble_mean = best_weights[0]*oof_xgb_preds + best_weights[1]*oof_catboost_preds + best_weights[2]*oof_nn_preds
test_ensemble_mean = best_weights[0]*test_xgb_preds + best_weights[1]*test_catboost_preds + best_weights[2]*test_nn_preds
print(f"\nOptimized Ensemble OOF RMSE: ${np.sqrt(mean_squared_error(y_true, oof_ensemble_mean)):,.2f}")

print("\n--- Starting Block 5: Training the Ensemble Error Model ---")
error_target = np.abs(y_true - oof_ensemble_mean)
X_for_error = X.copy()
X_for_error['mean_ensemble_pred'] = oof_ensemble_mean
X_test_for_error = X_test.copy()
X_test_for_error['mean_ensemble_pred'] = test_ensemble_mean

# CRITICAL FIX #2: Ensure test columns match train columns EXACTLY before prediction.
X_test_for_error = X_test_for_error[X_for_error.columns]
print("CRITICAL FIX APPLIED: Test feature order aligned for the error model.")

# (Assuming Optuna tuning for the error model was done, using placeholder params)
best_params_error = {'eta': 0.02, 'max_depth': 7, 'subsample': 0.75, 'colsample_bytree': 0.6, 'lambda': 5.0, 'alpha': 0.5}

oof_error_preds, test_error_preds = np.zeros(len(X)), np.zeros(len(X_test))
for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error, grade_for_stratify)):
    print(f"  Training error model fold {fold+1}/{N_SPLITS}...")
    dtrain_err = xgb.DMatrix(X_for_error.iloc[train_idx], label=error_target.iloc[train_idx])
    dval_err = xgb.DMatrix(X_for_error.iloc[val_idx])
    bst_error = xgb.train(best_params_error, dtrain_err, num_boost_round=3000, evals=[(xgb.DMatrix(X_for_error.iloc[val_idx], error_target.iloc[val_idx]),'v')], early_stopping_rounds=100, verbose_eval=False)
    oof_error_preds[val_idx] = bst_error.predict(dval_err, iteration_range=(0, bst_error.best_iteration))
    test_error_preds += bst_error.predict(xgb.DMatrix(X_test_for_error), iteration_range=(0, bst_error.best_iteration)) / N_SPLITS

print(f"\nFinal Error Model OOF RMSE: ${np.sqrt(mean_squared_error(error_target, oof_error_preds)):,.2f}")


--- Starting Block 4: Optimizing 3-Model Ensemble Weights ---
Optimal Weights -> XGB: 0.409, CB: 0.489, NN: 0.102

Optimized Ensemble OOF RMSE: $94,957.93

--- Starting Block 5: Training the Ensemble Error Model ---
CRITICAL FIX APPLIED: Test feature order aligned for the error model.
  Training error model fold 1/5...
  Training error model fold 2/5...
  Training error model fold 3/5...
  Training error model fold 4/5...
  Training error model fold 5/5...

Final Error Model OOF RMSE: $60,337.91


In [11]:
# =============================================================================
# BLOCK 6: FINAL CALIBRATION AND SUBMISSION
# =============================================================================
print("\n--- Starting Block 6: Final Analysis, Calibration, and Submission ---")

# --- Step 1: Clip & Analyze Final Prediction Arrays ---
oof_error_final = np.clip(oof_error_preds, 0, None)
test_error_final = np.clip(test_error_preds, 0, None)

print("\n--- FINAL FORENSIC ANALYSIS ---")
print("Describing 'test_ensemble_mean':")
print(pd.Series(test_ensemble_mean).describe())
print("\nDescribing 'test_error__final':")
print(pd.Series(test_error_final).describe())

# --- Step 2: Calibrate Intervals via Grid Search ---
print("\n--- Step 2: Calibrating intervals with Winkler Score... ---")
best_score, best_a, best_b = float('inf'), 1.0, 1.0
for a in np.arange(1.8, 2.8, 0.01):
    for b in np.arange(1.8, 2.8, 0.01):
        score = winkler_score(y_true, oof_ensemble_mean - oof_error_final * a, oof_ensemble_mean + oof_error_final * b)
        if score < best_score:
            best_score, best_a, best_b = score, a, b

# --- Step 3: Create and Verify Submission ---
print("\n--- Step 3: Creating and Verifying Submission File ---")
final_lower = test_ensemble_mean - test_error_final * best_a
final_upper = test_ensemble_mean + test_error_final * b
final_upper = np.maximum(final_lower, final_upper)

submission_df = pd.DataFrame({'id': pd.read_csv(os.path.join(DATA_PATH,'test.csv'))['id'], 'pi_lower': final_lower, 'pi_upper': final_upper})

if submission_df.isnull().sum().any():
    print("\n\n*** CRITICAL WARNING: Null values detected in the final submission DataFrame! ***\n")
else:
    print("\nSUCCESS: Final submission DataFrame is clean and ready.")




--- Starting Block 6: Final Analysis, Calibration, and Submission ---

--- FINAL FORENSIC ANALYSIS ---
Describing 'test_ensemble_mean':
count    2.000000e+05
mean    -8.652765e+06
std      4.134777e+09
min     -1.849128e+12
25%      3.134251e+05
50%      4.716368e+05
75%      7.320310e+05
max      2.998222e+06
dtype: float64

Describing 'test_error__final':
count    200000.000000
mean      53095.256415
std       52732.717250
min        5682.654541
25%       22500.658569
50%       35778.490967
75%       61006.354980
max      831254.953125
dtype: float64

--- Step 2: Calibrating intervals with Winkler Score... ---

--- Step 3: Creating and Verifying Submission File ---

SUCCESS: Final submission DataFrame is clean and ready.


In [ ]:
# --- Step 4: Save Submission and Print Final Results ---
print("\n" + "="*60)
print("--- FINAL RESULTS & SUBMISSION ---")
print("="*60)
print(f"Best OOF Winkler Score: {best_score:,.2f}")
print(f"Optimal Multipliers: a={best_a:.3f}, b={best_b:.3f}")

submission_filename = f'submission_final_3model_winkler_{int(best_score)}.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
print("\nSubmission Head:\n", submission_df.head())